# 06c: 亚群 Subset 重分析

面向 **非计算机专业 PI/学生**（ADR-0009）。

当 PI 在 06 全局注释后希望对某类细胞（如所有 T 细胞 / 上皮细胞 /
SPEM 谱系）做更精细的亚群分析时，本 notebook 执行以下流程：

1. **子集抽取**：研究者在 `SUBSET_LABELS` 中勾选上游 final 标签，不执行表达式
2. **03 重跑**：对子集重新选 HVG（全局 HVG 对亚群未必最优）
3. **04 重跑**：对子集重新做嵌入（子集的批次效应/生物变异结构不同）
4. **05 重跑**：对子集重新聚类（子集通常需要更细的分辨率）
5. **06 重跑**：对子集重新标注（精细细胞亚型，如 T_cell → CD4_Tcm/CD8_Tem）
6. **标签回流**：将精细标签写回派生的 `SUBSET_LABEL_COL`——
   **子集内的细胞**获得精细标签，**子集外的细胞保留 NaN**。
   `UPSTREAM_LABEL_COL` **不覆盖**；粗、细和统一标签分别由三个版本字段派生。

**为什么需要亚群重分析？**
全局分析的目标是区分大类（上皮 vs 免疫 vs 间质），参数为这个目标校准。
但同一大类内部的异质性（如 T 细胞的 CD4/CD8 亚群、上皮的 pit/neck/SPEM 梯度）
需要重新选 HVG + 重新聚类才能看清——这是 scRNA-seq 分析的标准实践，
不是"重跑浪费算力"。

**溯源机制**：`adata_sub.uns["subset_of"]` 记录来源 h5ad，
`adata_sub.uns["subset_filter"]` 记录筛选表达式，
任何下游分析都能追溯这个子集是如何产生的（见 SPEC 196）。

**实现纪律（ADR-0003/0009）**：直接调 scanpy 原生 API，
无 plugin/registry/class。每个 stage 是独立的 cell block，
非 CS 学生按顺序读下来能看懂每一步在做什么。

## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：06（注释），读 `06_annotated_v*.h5ad`
- **下游**：07（下游分析），产出两个文件：
  - `06c_subset_v*.h5ad`（子集独立 h5ad）
  - `06_annotated_<MAIN_OUTPUT_VERSION>.h5ad`（含派生的 `SUBSET_LABEL_COL`）

### 为什么要迭代回跑？
亚群重分析的质量取决于上游注释的准确性 + 子集筛选的合理性 + 子集内参数的选择。
当以下任一情况发生时需要重跑：
- PI 发现子集注释不够精细（如 T 细胞应拆成 CD4/CD8/Treg 但未拆）
- PI 调整了 `UPSTREAM_LABEL_COL`（上游 06 重跑后）
- PI 想分析另一类细胞（改 `SUBSET_LABELS`）
- 子集 Leiden 聚类结果不理想（改 `RESOLUTIONS` 或 `N_PCS`）
- 标记物 CSV 更新（改 `MARKER_CSV`）

PI 的原始要求（来自项目构思）：
> "注释这一步，也可能在注释的过程中发现前面高可变基因的选择、embedding 的构建，
> 还有分群的参数等等需要调整，应可以随时调回去重跑一些流程，需要建立这种循环不断迭代的机制。"

### 如何回跑（三步操作）
1. **改 `UPSTREAM_RUN_ID`**——选择已提升且校验通过的 06 run
2. **分别改三个版本字段**——上游标签、subset 产物和主对象回流互不混用
3. **调整参数**（在下方 `# === PARAMS ===` 区域改 `SUBSET_LABELS`、`N_PCS`、
   `RESOLUTIONS` 或标记物参数）
   → 重跑本 notebook（Cell → Run All）

### 版本约定
- 三个版本字段独立递增，避免上游标签、subset 和主对象回流错配。
  旧版 `.h5ad` 文件**不覆盖不删除**，保留在 `results/` 目录供追溯对比。
- **`experimental`**：刚跑出、尚未经 PI 审查确认的版本（默认值）。
- **`promoted`**：PI 审查后认为子集分析质量可接受、可传给下游使用的正式版本。

### 两个输出对象的溯源策略（重要）
本 notebook 产出两个 h5ad，但**区别对待**：
- **`adata_sub`（子集对象）**：全新对象，独立顶层追踪字段
  （`stage="06c_subset"`、独立版本字段、`upstream=[validated checkpoint]`、`status="experimental"`）。
  保留 `06c_subset_<SUBSET_OUTPUT_VERSION>` 嵌套 dict 作为细节记录。
- **`main_adata`（主对象回流更新）**：本身是 06 的产物，**不覆盖**
  其顶层 `stage` / `version` / `upstream` 字段——那会篡改 06 的溯源链。
  `main_adata` 仅写入 `06c_subset_reflow_<MAIN_OUTPUT_VERSION>` 记录回流。
  主对象的追踪字段由 06 负责维护。

### 追溯链
如需查询"06c 有哪些版本？"或"这个子集依赖哪个 06 版本？"，
可在 Python 中检查子集 h5ad 的 `adata.uns["stage"]` / `adata.uns["upstream"]`。


In [ ]:
# === PARAMS ===
# 上游必须是已提升且校验通过的 06_annotated run；标签与输出版本彼此独立。
# N_TOP_GENES         -- HVG 数量
# N_PCS               -- PCA 主成分数
# N_PCS_USE           -- 实际送入邻居图/Harmony 的 PC 数（与 04/05 一致）
# RESOLUTIONS         -- Leiden 多分辨率列表
# MARKER_CSV          -- 标记物知识库 CSV
# RANDOM_SEED         -- 随机种子（全流程一致）

UPSTREAM_RUN_ROOT = "results/runs"
UPSTREAM_RUN_ID = "06-annotated-v1-run001"
RUN_ROOT = "results/runs"
RUN_ID = "06c-subset-v1-run001"
UPSTREAM_LABEL_VERSION = "v1"
SUBSET_OUTPUT_VERSION = "v1"
MAIN_OUTPUT_VERSION = "v2"
SUBSET_LABELS = ["CD4 T", "CD8 T", "Treg", "NK cell", "B cell"]
MIN_SUBSET_CELLS = 50

# ---- 分析参数 ----
N_TOP_GENES = 3000
N_PCS       = 50
N_PCS_USE   = 30  # 实际送入邻居图/Harmony 的 PC 数（与 04/05 一致）
RESOLUTIONS = [0.4, 0.6, 0.8, 1.0, 1.2, 1.6]
MARKER_CSV  = "references/markers/gastric_TEST_markers.csv"

# ---- LLM（统一 llm_config 路由，从 .env LLM_GROUP* 读取）----
MLLM_ENABLED = True
MLLM_MODELS = None          # None = 从 .env 自动构建；手动指定如 ["claude-3-5-haiku-20241022"]
MLLM_CONSENSUS_THRESHOLD = 0.7
MLLM_ENTROPY_THRESHOLD = 0.5   # 子集模式簇数少、标签空间窄，模型分歧天然更大，比全局 0.3 适当放宽
MLLM_MAX_DISCUSSION_ROUNDS = 3

RANDOM_SEED = 42

# === PI 确认闸门控制（决策6：subset 注释闸门 + 回流闸门，与 06_annotated 语义一致） ===
# subset 注释闸门（SUBSET_PI_CONFIRMED）：subset 内精细注释同样须 PI 显式确认后才能落 final。
#   LLM/marker 建议只作为参考——涉及下游分析的全部解释，必须由 PI 逐簇确认。
#   SUBSET_PI_CONFIRMED=False 时只产出 suggested + pi_confirmed 层，绝不创建 cell_type_final_subset 列。
# 回流闸门（MAIN_REFLOW_CONFIRMED）：将 subset 精细标签写回主对象是破坏性操作
#   （修改主对象 obs 列），须 PI 单独确认 + 主版本 MAIN_OUTPUT_VERSION 刻意 bump。
#   两个闸门独立：可先确认 subset 注释（subset 有 final）但暂不回流主对象。
SUBSET_PI_CONFIRMED = False  # subset 注释闸门总开关；False 时永远只写 draft、不写 final
MAIN_REFLOW_CONFIRMED = False  # 回流闸门总开关；False 时主对象不加载/不改动
SUBSET_ACCEPT_ALL_SUGGESTED = False  # "全部接受建议"批量操作；仅当 SUBSET_PI_CONFIRMED=True 时，
                                      # 才从 LLM suggested 生成逐簇 accept 审计记录
subset_final_gate_passed = False  # subset 注释闸门结果标记变量，由 gate cell 赋值；
                                  # PARAMS 先给确定初值。禁 dir() 流控残留
main_reflow_gate_passed = False  # 回流闸门结果标记变量，由回流 gate cell 赋值

# === PI 确认输入方式（6 列 schema，与 06_annotated 一致） ===
# 支持两种方式：
#   方式 1（推荐非 CS 用户）：CSV 文件，6 列
#     cluster, suggested_label, pi_final_label, decision(accept|modify|unresolved), note, annotation_version
#     在 SUBSET_PI_CONFIRMATION_CSV 中设文件路径，然后在 Excel/WPS 中编辑即可
#   方式 2（程序员）：直接编辑下方 pi_subset_decisions Python 字典
SUBSET_PI_CONFIRMATION_CSV = ""  # 留空则使用 Python dict；填路径则从 CSV 读取

# 每运行一次 06c 递增 N
# 如 T cells 做了一次，上皮又做了一次 → N 分别 = 1, 2

In [ ]:
# 确保框架 src/ 在 sys.path 并切换到项目根目录。
import sys, os, gc, datetime, warnings, json, re
from pathlib import Path
_root = os.getcwd()
# 向上逐级查找项目根（含 src/scrna_integration 的目录），兼容任意嵌套深度
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
        break
    _root = os.path.dirname(_root)
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
print(f"PROJECT_ROOT: {_root}")

# A800 64核 OpenBLAS默认全开致线程爆炸（200+线程冻结），限制为4
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMBA_NUM_THREADS", "4")

# 导入（scanpy 原生 + 框架函数）
import scanpy as sc
import scvi
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scrna_integration.run_contract import (
    atomic_write_json, collect_runtime_provenance, prepare_run, resume_run, sha256_file,
    snapshot_effective_parameters, validate_artifacts, validate_checkpoint,
)

# 标记物库加载器（替代原来的 from scrna_integration import load_markers）
# src/notebook 边界铁律（2026-07-10 PI 定稿）：分析逻辑留 notebook cell，可见可调可开关
def load_markers(csv_path, roles=("canonical", "optional")):
    """加载标记物 CSV，按 cell_type 分组并按 role 过滤。

    CSV 格式: tissue, cell_type, marker, role, reference, notes
    Args:
        csv_path: CSV 文件路径
        roles: role 过滤元组，默认 ("canonical", "optional")；None 返回完整三层字典
    Returns: 细胞类型 → 标记物列表（扁平）或 细胞类型 → {role: [...]}（嵌套）
    """
    df = pd.read_csv(csv_path, comment="#")
    _required = ["cell_type", "marker", "role"]
    _missing = [col for col in _required if col not in df.columns]
    if _missing:
        raise ValueError(
            f"marker CSV 缺少必需列: {', '.join(_missing)}；"
            f"需要的列: {', '.join(_required)}。"
            f"当前文件列: {', '.join(df.columns.tolist())}。"
            f"请检查 CSV 列名是否与模板一致（区分大小写）。"
        )
    if isinstance(roles, str):
        raise TypeError(
            "roles 参数必须为 list 或 tuple，不能传字符串。"
            "若只想查一种 role，请写成 ('canonical',) 注意末尾逗号。"
        )
    if roles is None:
        result = {}
        for role in ("canonical", "optional", "negative"):
            role_df = df[df["role"] == role]
            for ct, group in role_df.groupby("cell_type"):
                result.setdefault(ct, {}).setdefault(role, [])
                result[ct][role] = group["marker"].tolist()
        for ct in result:
            for role in ("canonical", "optional", "negative"):
                result[ct].setdefault(role, [])
        return result
    filtered = df[df["role"].isin(roles)]
    return {ct: group["marker"].tolist() for ct, group in filtered.groupby("cell_type")}

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
warnings.filterwarnings("ignore", category=FutureWarning)

# 固定随机种子（全流程一致，保证可复现）
np.random.seed(RANDOM_SEED)

_VERSION_RE = re.compile(r"[A-Za-z0-9][A-Za-z0-9._-]*\Z")
UPSTREAM_LABEL_COL = f"cell_type_final_{UPSTREAM_LABEL_VERSION}"
SUBSET_LABEL_COL = f"cell_type_final_subset_{SUBSET_OUTPUT_VERSION}"
MAIN_LABEL_COL = f"cell_type_unified_{MAIN_OUTPUT_VERSION}"

# subset 注释字段族（Vs=SUBSET_OUTPUT_VERSION，全部由 f-string 单点生成，禁硬编码字面量）
# 这些列名与 06_annotated 的字段族对齐：marker → llm_suggested → pi_confirmed → final
SUBSET_MARKER_COL = f"cell_type_marker_subset_{SUBSET_OUTPUT_VERSION}"
SUBSET_LLM_COL = f"cell_type_llm_suggested_subset_{SUBSET_OUTPUT_VERSION}"
SUBSET_PI_CONFIRMED_COL = f"cell_type_pi_confirmed_subset_{SUBSET_OUTPUT_VERSION}"
SUBSET_PROV_KEY = f"annotation_provenance_subset_{SUBSET_OUTPUT_VERSION}"

# 安全排序 key：处理簇 ID 可能为非数值字符串（如 "Chief cell"）
def _safe_sort_key(x):
    try:
        return (0, int(x))
    except (ValueError, TypeError):
        return (1, str(x))

def _load_verified_upstream():
    upstream = resume_run(UPSTREAM_RUN_ROOT, UPSTREAM_RUN_ID, promoted=True)
    manifest_path = upstream.promoted_dir / "manifest.json"
    checkpoint = validate_checkpoint(manifest_path); validate_artifacts(manifest_path)
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("run_id") != UPSTREAM_RUN_ID or manifest.get("stage") != "06_annotated": raise ValueError("上游必须是匹配 RUN_ID 的 06_annotated run")
    status = manifest.get("stage_status")
    if status == "SUCCESS_WITH_WARNINGS":
        accepted = manifest.get("warning_acceptance")
        if not isinstance(accepted, dict) or not all(isinstance(accepted.get(k), str) and accepted[k].strip() for k in ("accepted_by", "accepted_at")): raise ValueError("上游 warnings 尚未被研究者有效接受")
    elif status != "SUCCESS": raise ValueError(f"上游 stage_status 不可消费: {status!r}")
    manifest_sha256, checkpoint_sha256 = sha256_file(manifest_path), sha256_file(checkpoint)
    return sc.read_h5ad(checkpoint), checkpoint, manifest_path, manifest_sha256, checkpoint_sha256

def _subset_preflight(current):
    errors = []
    versions = (UPSTREAM_LABEL_VERSION, SUBSET_OUTPUT_VERSION, MAIN_OUTPUT_VERSION)
    versions_valid = all(isinstance(v, str) and _VERSION_RE.fullmatch(v) for v in versions)
    labels_valid = isinstance(SUBSET_LABELS, (list, tuple)) and bool(SUBSET_LABELS) and all(isinstance(x, str) and x.strip() for x in SUBSET_LABELS) and len(set(SUBSET_LABELS)) == len(SUBSET_LABELS)
    if not versions_valid: errors.append("版本号必须是安全字符串")
    if UPSTREAM_LABEL_COL not in current.obs: errors.append(f"缺少 {UPSTREAM_LABEL_COL}")
    else:
        labels = current.obs[UPSTREAM_LABEL_COL]
        if labels.isna().any() or not labels.astype(str).str.strip().ne("").all(): errors.append("final label 必须完整且非空")
    if not labels_valid: errors.append("SUBSET_LABELS 必须是非空、唯一的非空字符串 list/tuple")
    minimum_valid = isinstance(MIN_SUBSET_CELLS, int) and not isinstance(MIN_SUBSET_CELLS, bool) and MIN_SUBSET_CELLS >= 1
    mask = pd.Series(False, index=current.obs_names, dtype=bool)
    if versions_valid and labels_valid and minimum_valid and UPSTREAM_LABEL_COL in current.obs:
        available = set(current.obs[UPSTREAM_LABEL_COL].dropna())
        if not set(SUBSET_LABELS).issubset(available): errors.append("SUBSET_LABELS 含上游不存在的标签")
        else: mask = current.obs[UPSTREAM_LABEL_COL].isin(SUBSET_LABELS)
    if not mask.index.equals(current.obs_names) or mask.dtype != bool: errors.append("subset mask 必须与 obs index 对齐且为 boolean")
    n = int(mask.sum())
    if not minimum_valid or n < MIN_SUBSET_CELLS or n >= current.n_obs: errors.append("subset 细胞数必须满足整数 MIN_SUBSET_CELLS <= n < total")
    return errors, mask

def _write_preflight_failure(errors):
    failed_run = prepare_run(RUN_ROOT, RUN_ID)
    failed = {"run_id": RUN_ID, "stage": "06c_subset", "stage_status": "FAILED", "effective_parameters": snapshot_effective_parameters(globals(), path_root=_root),
              "runtime_provenance": collect_runtime_provenance(_root, ("scanpy", "anndata")), "inputs": [{"path": str(_upstream_manifest_path), "sha256": _upstream_manifest_sha256}, {"path": str(_upstream_checkpoint), "sha256": _upstream_checkpoint_sha256}],
              "hard_postconditions": {"input_preflight": False}, "failure": {"preflight": errors}}
    atomic_write_json(failed_run.manifest_path, failed); return failed_run

# 上游验证、hash 校验和 h5ad 读取失败时，不创建本次 06c run。
adata, _upstream_checkpoint, _upstream_manifest_path, _upstream_manifest_sha256, _upstream_checkpoint_sha256 = _load_verified_upstream()
_preflight_errors, _mask = _subset_preflight(adata)
if _preflight_errors:
    _write_preflight_failure(_preflight_errors); raise ValueError(f"Stage 06c preflight FAILED: {_preflight_errors}")

# 合法输入只准备内存中的 mask；本 PR 不为成功运行创建 current run。
os.makedirs("results/figures/06c_subset", exist_ok=True)
_verified_upstream_index = adata.obs_names.copy()
_verified_upstream_coarse = adata.obs[UPSTREAM_LABEL_COL].copy()
_verified_upstream_obs_columns = tuple(adata.obs.columns)
print(f"已验证上游: {_upstream_checkpoint} ({adata.n_obs:,} cells)")

## 子集抽取

按 `SUBSET_LABELS` 与派生 final 标签列构造 boolean mask，`copy()` 保证子集独立。
在子集 `uns` 中记录 `subset_of` 和 `subset_labels` 以供溯源。

**为什么用 `.copy()` 而非 view？**
后面的 HVG 重选、归一化、嵌入都需要独立的内存空间。
view 会意外修改主 adata 的数据，copy 隔离这个风险。

**为什么先确认 `UPSTREAM_LABEL_COL` 存在？**
subset 筛选依赖于这个列——如果该列不存在（如 PI 还没拍板），
整条分析链路无意义。提前报错比后面发现好。

In [ ]:
# mask 已在任何算法消费前完成标签、index、dtype 和细胞数 preflight。
print(f"上游标签列: {UPSTREAM_LABEL_COL}; 选择标签: {SUBSET_LABELS}")
print(f"筛选前: {adata.n_obs:,} 细胞")
print(f"筛选后: {_mask.sum():,} 细胞 ({_mask.sum()/adata.n_obs*100:.1f}%)")

# copy() 保证独立内存空间
adata_sub = adata[_mask].copy()
print(f"\n子集 adata: {adata_sub.n_obs:,} 细胞 x {adata_sub.n_vars:,} 基因")

# ---- 溯源元数据 ----
# 为什么记录这些？任何下游分析或合作者拿到这个 h5ad 后，
# 通过 uns["subset_of"] 和 uns["subset_filter"] 就能追溯来源——
# 不需要翻 notebook 找参数。
adata_sub.uns["subset_of"] = str(_upstream_checkpoint)
adata_sub.uns["subset_labels"] = list(SUBSET_LABELS)
adata_sub.uns["subset_n_cells_before"] = adata.n_obs
adata_sub.uns["subset_n_cells_after"] = adata_sub.n_obs
print(f"溯源信息已记录: subset_of={_upstream_checkpoint}")
print(f"                   subset_labels={SUBSET_LABELS}")

# 释放主 adata（后续不需要它，直到回流步骤）
del adata
gc.collect()
print("主 adata 已释放")

## 03: 归一化 + 高变基因重选

**为什么子集要重选 HVG？**
全局 HVG 选的是能区分所有大类（上皮 vs 免疫 vs 间质）的基因。
但在 T 细胞子集中，这些基因大部分不表达（如上皮特异基因 MUC5AC），
真正区分 CD4/CD8/Treg 的基因（如 CD4/CD8A/FOXP3）在全局 HVG 中
可能因为跨大类差异不够大而被过滤掉。
因此子集分析的第一步永远是 **对子集重新选 HVG**。

**为什么先存 counts 层？**
归一化会覆盖 `adata.X`，但 06 的基因集评分等下游可能需要 raw counts。
存到 `layers["counts"]` 是 scanpy 标准实践，学生应该学会这个习惯。

In [ ]:
# 03: 归一化 + log + HVG（在子集上重跑）。
print("=== 03: normalize + HVG re-selection on subset ===")

# 保留 raw counts（归一化前）
# 为什么存 layers["counts"]？这是 scanpy 社区约定——
# 任何下游分析需要 raw counts 时从这里取，不用回头找 02 输出。
adata_sub.layers["counts"] = adata_sub.X.copy()

# 归一化到 10,000 counts per cell
# 为什么 target_sum=1e4？这是 scRNA-seq 的标准归一化目标——
# 与 10x Genomics 的默认值、Cell Ranger 的输出口径一致。
sc.pp.normalize_total(adata_sub, target_sum=1e4)
sc.pp.log1p(adata_sub)

# HVG 重选——用 seurat_v3 flavor
# 为什么 flavor="seurat_v3"？seurat_v3 基于方差稳定化变换选 HVG，
# 对比默认的 seurat（基于 dispersion），在子集分析中对稀有亚群的标记
# 基因检出率更高。见 ADR-0009 注释中文化时对 flavor 选择的讨论。
sc.pp.highly_variable_genes(
    adata_sub, n_top_genes=N_TOP_GENES, flavor="seurat_v3",
)
# 也可以额外指定 batch_key 避免批次驱动 HVG 选择：
# sc.pp.highly_variable_genes(
#     adata_sub, n_top_genes=N_TOP_GENES, flavor="seurat_v3",
#     batch_key="source_dataset",
# )

n_hvg = adata_sub.var["highly_variable"].sum()
print(f"\nHVG 选择: {n_hvg}/{adata_sub.n_vars} 基因标记为 highly_variable")
print(f"  前 10 个 HVG: {list(adata_sub.var_names[adata_sub.var['highly_variable']][:10])}")

# 可视化检查
sc.pl.highly_variable_genes(adata_sub, show=False)
plt.savefig("results/figures/06c_subset_hvg.png", dpi=120, bbox_inches="tight")
plt.close()
print("  HVG 图已保存: results/figures/06c_subset_hvg.png")

# 内存纪律：归一化不改变 sparse 性质，但确认 dtype
adata_sub.X = adata_sub.X.astype(np.float32)
assert sp.issparse(adata_sub.X) and adata_sub.X.dtype == np.float32
print("  内存自检: X sparse CSR float32 OK")

# 记录运行元数据
adata_sub.uns[f"normalize_{SUBSET_OUTPUT_VERSION}"] = {
    "target_sum": 1e4,
    "hvg_flavor": "seurat_v3",
    "n_top_genes": N_TOP_GENES,
    "n_hvg": int(n_hvg),
    "subset": True,  # 标记这是子集重跑
    "timestamp": datetime.datetime.now().isoformat(),
}

## 04: 多方法嵌入（在子集上）

与全局 04 相同的方法阵容，但在子集上独立运行。

**为什么子集需要重新嵌入？**
全局 PCA 的 loadings 由所有细胞的方差结构决定——在上皮细胞主导的
全局数据中，PC1-3 大概率是上皮 vs 免疫的差异。T 细胞子集的内部
变异（CD4 vs CD8、naive vs memory）在这些 PC 上可能是噪声。
在子集上重新跑 PCA+整合，嵌入空间才能真正反映子集内部的生物学变异。

**批次整合**：子集通常跨多个 source_dataset（病种/平台），
仍需要 Harmony/scVI 去批次。使用与全局相同的 `batch_key="source_dataset"`。

In [ ]:
# 04: PCA + Harmony + scVI（在子集上独立运行）。
print("=== 04: embedding on subset ===")

# ---- PCA ----
sc.tl.pca(adata_sub, n_comps=N_PCS, use_highly_variable=True, svd_solver="arpack")
print(f"PCA 完成: {N_PCS} PCs")

# ---- Harmony 去批次（harmonypy 直调，避免 scanpy wrapper >=2.0 维度 bug）----
# 为什么 batch_key="source_dataset"？不同数据集可能存在不同的技术/平台
# 批效应。Harmony 在 PCA 空间做 soft-clustering 对齐，
# 对 scRNA-seq 的稀疏数据比 MNN/CCA 更快且不易过校正。
# 为什么直调 harmonypy？sc.external.pp.harmony_integrate 在 harmonypy >=2.0
# 有维度兼容问题——直调避免中间层版本耦合。
import harmonypy

ho = harmonypy.run_harmony(
    adata_sub.obsm["X_pca"][:, :N_PCS_USE],  # 只用前 N_PCS_USE 个 PC
    adata_sub.obs,
    "source_dataset",
    max_iter_harmony=20,
    random_state=RANDOM_SEED,
)
adata_sub.obsm["X_pca_harmony"] = ho.Z_corr
assert ho.Z_corr.shape == (adata_sub.n_obs, N_PCS_USE), (
    f"Harmony shape 不符: {ho.Z_corr.shape} != ({adata_sub.n_obs}, {N_PCS_USE})")
print(f"✓ Harmony 完成: {adata_sub.obsm['X_pca_harmony'].shape}")

# 收敛检查
if hasattr(ho, 'check_convergence'):
    _converged = ho.check_convergence()
elif hasattr(ho, 'converged'):
    _c = ho.converged
    _converged = _c() if callable(_c) else _c
else:
    _converged = True

if not _converged:
    print(f"⚠️ Harmony 未收敛——考虑增大 max_iter 或检查子集内批次结构")

# ---- scVI（深度生成模型）----
# scVI 假设计数数据服从零膨胀负二项分布，从 raw counts 学习隐变量。
# 为什么用 counts 层而非 normalized X？scVI 内部有自己的归一化机制，
# 直接用 counts 避免丢失分布信息。
scvi.model.SCVI.setup_anndata(
    adata_sub, batch_key="source_dataset", layer="counts",
)
_model = scvi.model.SCVI(adata_sub, n_latent=30, n_layers=2)
# max_epochs 可根据子集大小调整——子集小（<5000 cells）用较少 epoch
_max_epochs = 200 if adata_sub.n_obs > 5000 else 100
_model.train(max_epochs=_max_epochs, early_stopping=True)
adata_sub.obsm["X_scVI"] = _model.get_latent_representation()
print(f"scVI 完成: obsm['X_scVI'] shape={adata_sub.obsm['X_scVI'].shape}")

# 记录元数据
adata_sub.uns[f"embedding_{SUBSET_OUTPUT_VERSION}"] = {
    "methods": ["pca", "harmony", "scvi"],
    "n_pcs": N_PCS,
    "n_latent": 30,
    "batch_key": "source_dataset",
    "subset": True,
    "timestamp": datetime.datetime.now().isoformat(),
}
print("  嵌入元数据已记录")

## 05: 多分辨率聚类（在子集上）

用 Harmony 嵌入构建 kNN 图，然后做多分辨率 Leiden 聚类。
**为什么用 Harmony embedding 建图？** Harmony 已去除已知批次效应，
在此基础上建图能减少"同细胞类型因技术差异被拆成不同簇"的问题。

**为什么在子集上用更细的分辨率（1.0–1.6）？**
子集中的生物学差异比全局更 subtle——CD4 Tcm vs CD4 Tem 之间的
标记基因差异远小于 T cell vs epithelial 之间的差异。
更细的分辨率让 Leiden 能捕捉这些微妙结构。

PI 在多个分辨率中选最合理的，方法与全局 05 相同。

In [ ]:
# 05: 多分辨率 Leiden 聚类（在子集上）。
print("=== 05: clustering on subset ===")

# 用 Harmony 嵌入构建 kNN 图
# 为什么 use_rep="X_pca_harmony"？去批次后再建邻居关系，
# 避免"相同生物学但因批次差异被归为不同簇"。
sc.pp.neighbors(adata_sub, use_rep="X_pca_harmony", n_neighbors=15,
                n_pcs=N_PCS_USE, random_state=RANDOM_SEED)

# 计算 UMAP（供后续可视化）
sc.tl.umap(adata_sub)
print(f"UMAP 完成: obsm['X_umap']")

# 多分辨率 Leiden clustering
for res in RESOLUTIONS:
    key = f"leiden_res_{res}"
    sc.tl.leiden(adata_sub, resolution=res, key_added=key)
    n_clusters = adata_sub.obs[key].nunique()
    print(f"  leiden_res_{res}: {n_clusters} 簇")

print(f"\n多分辨率聚类完成。{len(RESOLUTIONS)} 个分辨率已写入 obs。")

# 可视化各分辨率
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for ax, res in zip(axes, RESOLUTIONS):
    sc.pl.umap(
        adata_sub, color=f"leiden_res_{res}", ax=ax,
        title=f"Leiden res={res}", legend_loc="right margin",
        show=False,
    )
for ax in axes[len(RESOLUTIONS):]:
    ax.set_visible(False)
fig.tight_layout()
fig.savefig("results/figures/06c_subset_leiden_sweep.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print("  Leiden sweep 图已保存: results/figures/06c_subset_leiden_sweep.png")

# 记录元数据
adata_sub.uns[f"clustering_{SUBSET_OUTPUT_VERSION}"] = {
    "use_rep": "X_pca_harmony",
    "resolutions": RESOLUTIONS,
    "n_neighbors": 15,
    "subset": True,
    "timestamp": datetime.datetime.now().isoformat(),
}

### PI 选择子集聚类分辨率

多分辨率中 PI 选择一个最合理的分辨率作为本子集的标签列。
通常的原则是：簇数不能太多（过度分裂）也不能太少（欠聚类），
要能在 UMAP 上看到清晰的群体分离。

In [ ]:
# === PI 选择子集聚类分辨率 ===
# PI 查看上方的 Leiden sweep UMAP 图后，选择一个分辨率。
# 为什么让 PI 选？自动选分辨率的算法（如 silhouette 最大值）
# 对子集分析的 subtle 结构不够敏感——PI 的领域知识在这里不可替代。
LEIDEN_COL = "leiden_res_0.6"  # PI 从上方图中选一个最合理的分辨率

if LEIDEN_COL not in adata_sub.obs.columns:
    print(f"⚠ 警告: '{LEIDEN_COL}' 不在 obs 中。可用列: "
          f"{[c for c in adata_sub.obs.columns if c.startswith('leiden_')]}")
else:
    n_clust = adata_sub.obs[LEIDEN_COL].nunique()
    print(f"选用聚类列: {LEIDEN_COL} ({n_clust} 簇)")
    # 确保是 categorical
    if not pd.api.types.is_categorical_dtype(adata_sub.obs[LEIDEN_COL]):
        adata_sub.obs[LEIDEN_COL] = adata_sub.obs[LEIDEN_COL].astype("category")

## 06: 子集重标注（精细细胞亚型）

与全局 06 相同的多方法标注流程，但目标改为精细亚型：
- 全局 06 标注的是：T_cell, B_cell, pit_cell, SPEM...
- 子集 06 标注的是：CD4_Tcm, CD8_Tem, Treg, MAIT...（在 T 细胞子集内）

**为什么用多方法？**
与全局 06 相同的理由——单方法容易出错，多方法交叉比对才能
找到置信度高的标签。但这里调用 mLLMCelltype 时，tissue prompt
应调整为子集对应的 context（如 "human peripheral blood T cells"
而非 "human stomach"）。

**注意**：本 cell 的 LLM 调用同样受 key 守卫——无 key 时优雅跳过。

In [ ]:
# 06: 子集重标注（精细细胞亚型）。
print("=== 06: re-annotation on subset (finer cell types) ===")

# ---- B4：应用 mLLMCelltype monkey-patch（与 06 统一，修复库已知缺陷） ----
from scrna_integration.llm_config import (
    apply_mllmcelltype_patches,
    build_mllmcelltype_config,
)
_patch_ok = apply_mllmcelltype_patches(
    max_retries=3,
    retry_delay=2,
    timeout=120,
    max_tokens_override=16384,
)
if _patch_ok:
    print("mLLMCelltype patch 已应用")
else:
    print("⚠️ mLLMCelltype patch 失败，LLM 注释将跳过")

# ---- 方法 1: 标记物 dotplot（PI 手动标注）----
try:
    _markers = load_markers(MARKER_CSV)
    print(f"✓ 标记物库加载: {MARKER_CSV} ({sum(len(v) for v in _markers.values())} 基因)")
except FileNotFoundError:
    print(f"⚠️ 标记物文件不存在: {MARKER_CSV}——跳过 dotplot 和基因集评分")
    _markers = {}

# === Marker 库与 subset 类型一致性检查 ===
# 根据 SUBSET_LABELS 推断免疫/上皮类型，与 MARKER_CSV 做一致性检查

_marker_filename = os.path.basename(MARKER_CSV).lower()

# B4：从子集筛选表达式推导 tissue context（代替硬编码 "human gastric mucosa"）
_immune_keywords = {"immune", "t_cell", "b_cell", "cd4", "cd8", "treg", "nk", "myeloid",
                    "t cell", "b cell", "nk cell"}
_epithelial_keywords = {"epithelial", "chief", "parietal", "mucous", "spem", "pit"}

_filter_lower = " ".join(SUBSET_LABELS).lower()
_filter_is_immune = any(kw in _filter_lower for kw in _immune_keywords)
_filter_is_epithelial = any(kw in _filter_lower for kw in _epithelial_keywords)

if _filter_is_immune:
    _tissue_context = "human immune cells"
    print(f"子集组织上下文: {_tissue_context}（从 SUBSET_LABELS 免疫关键词推导）")
elif _filter_is_epithelial:
    _tissue_context = "human gastric epithelium"
    print(f"子集组织上下文: {_tissue_context}（从 SUBSET_LABELS 上皮关键词推导）")
else:
    _tissue_context = "human gastric mucosa"
    print(f"子集组织上下文: {_tissue_context}（默认，无法从 SUBSET_LABELS 推导）")

if _filter_is_immune and "epithelial" in _marker_filename:
    print("⚠️ SUBSET_LABELS 选择的是免疫细胞，但 MARKER_CSV 是上皮标记物！")
    print(f"   SUBSET_LABELS: {SUBSET_LABELS}")
    print(f"   MARKER_CSV: {MARKER_CSV}")
    print("   -> 建议改为免疫标记物 CSV（如 gastric_immune.csv），否则 dotplot/评分无意义")
elif _filter_is_epithelial and "immune" in _marker_filename:
    print("⚠️ SUBSET_LABELS 选择的是上皮细胞，但 MARKER_CSV 是免疫标记物！")
    print(f"   -> 建议改为上皮标记物 CSV")
else:
    print(f"Marker 库与 subset 类型一致性检查通过")
    print(f"   SUBSET_LABELS 关键词: {_filter_lower[:80]}...")
    print(f"   MARKER_CSV: {MARKER_CSV}")

_all_marker_genes = sorted(set(g for glist in _markers.values() for g in glist))
_avail = [g for g in _all_marker_genes if g in adata_sub.var_names]
_avail = _avail[:30]  # 子集分析减少展示基因数避免图过大

if _avail and LEIDEN_COL in adata_sub.obs.columns:
    sc.pl.dotplot(
        adata_sub, var_names=_avail, groupby=LEIDEN_COL,
        dendrogram=True, standard_scale="var",
        title=f"Canonical marker dotplot (subset, {LEIDEN_COL})",
        show=False,
    )
    plt.savefig("results/figures/06c_subset_dotplot.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("  标记物 dotplot 已保存: results/figures/06c_subset_dotplot.png")

# ---- 方法 2: 基因集评分（scanpy score_genes）----
_score_cols = []
for ct, gene_list in _markers.items():
    _present = [g for g in gene_list if g in adata_sub.var_names]
    if len(_present) < 2:
        continue
    col = f"score_{ct}"
    sc.tl.score_genes(
        adata_sub, gene_list=_present, score_name=col,
        ctrl_size=max(1, min(len(_present), 50)),
    )
    _score_cols.append(col)
print(f"  基因集评分: {len(_score_cols)} 个评分列 -> obs")

# === mLLMCelltype 多模型共识注释（子集）===
# B4：调用签名与 06 统一（marker_genes + species 模式，非 adata + cluster_key）
# B7：默认多模型共识，单模型时诚实降级
if MLLM_ENABLED and _patch_ok:
    try:
        from mllmcelltype import interactive_consensus_annotation

        _api_keys, _base_urls, _model_list = build_mllmcelltype_config(
            project_root=_root,
            model_list_override=MLLM_MODELS,
        )

        if _model_list:
            _n_models = len(_model_list)
            _use_discussion = _n_models >= 2

            if not _use_discussion:
                print(f"⚠️ 仅配置 {_n_models} 个模型，多模型共识降级为单模型注释")
                print(f"  -> 单一模型标签未经交叉验证，置信度低于多模型共识")
                print(f"  -> 如需多模型共识，请在 .env 中配置多个 LLM_GROUP* 或同一 group 的多档模型")

            print(f"mLLMCelltype 模型: {_model_list}")
            print(f"开始子集注释（{_n_models} 模型，{'讨论模式' if _use_discussion else '单模型模式'}）...")

            # B4：统一调用签名——marker_genes + species 模式（与 06 一致）
            # DEG for mLLMCelltype
            if "rank_genes_06c" not in adata_sub.uns:
                sc.tl.rank_genes_groups(
                    adata_sub, groupby=LEIDEN_COL, method="wilcoxon",
                    n_genes=30, key_added="rank_genes_06c",
                )

            _rgg = adata_sub.uns["rank_genes_06c"]
            _cluster_names = list(_rgg["names"].dtype.names)
            _marker_genes = {}
            for cl in _cluster_names:
                _marker_genes[str(cl)] = _rgg["names"][cl][:10].tolist()

            _result = interactive_consensus_annotation(
                marker_genes=_marker_genes,
                species="human",
                models=_model_list,
                api_keys=_api_keys,
                base_urls=_base_urls if _base_urls else None,
                tissue=_tissue_context,  # B4：子集实际组织上下文
                consensus_threshold=MLLM_CONSENSUS_THRESHOLD,
                entropy_threshold=MLLM_ENTROPY_THRESHOLD,
                max_discussion_rounds=(
                    MLLM_MAX_DISCUSSION_ROUNDS if _use_discussion else 0
                ),
                use_cache=False,
            )

            # 列名改用 SUBSET_LLM_COL（cell_type_llm_suggested_subset_{Vs}，与 06 对齐）
            _llm_col = SUBSET_LLM_COL
            if hasattr(_result, "cell_types") and _result.cell_types:
                _llm_map = _result.cell_types
                adata_sub.obs[_llm_col] = (
                    adata_sub.obs[LEIDEN_COL].astype(str).map(_llm_map)
                ).astype("category")
                print(f"mLLMCelltype 子集注释完成: {len(_llm_map)} 簇 -> obs['{_llm_col}']")
            elif isinstance(_result, dict) and "consensus" in _result:
                _consensus = _result["consensus"]
                adata_sub.obs[_llm_col] = (
                    adata_sub.obs[LEIDEN_COL].astype(str).map(_consensus)
                ).astype("category")
                print(f"mLLMCelltype 子集注释完成: {len(_consensus)} 簇 -> obs['{_llm_col}']")
            elif isinstance(_result, dict):
                adata_sub.obs[_llm_col] = (
                    adata_sub.obs[LEIDEN_COL].astype(str).map(_result)
                ).astype("category")
                print(f"mLLMCelltype 子集注释完成 (dict): {len(_result)} 簇 -> obs['{_llm_col}']")
            else:
                print(f"mLLMCelltype 返回格式异常: {type(_result).__name__}")
        else:
            print("无可用模型，跳过 mLLMCelltype（.env 未配置 LLM_GROUP*）")
    except ImportError:
        print("mLLMCelltype 未安装 (pip install mllmcelltype)")
    except Exception as e:
        print(f"mLLMCelltype 异常: {e}")
        import traceback
        traceback.print_exc()
else:
    if not MLLM_ENABLED:
        print("MLLM_ENABLED=False，跳过")
    else:
        print("mLLMCelltype patch 失败，跳过 LLM 注释")

# 确保 SUBSET_LLM_COL 列存在（LLM 未跑或跳过时预建空列）
if SUBSET_LLM_COL not in adata_sub.obs.columns:
    adata_sub.obs[SUBSET_LLM_COL] = pd.Categorical([np.nan] * adata_sub.n_obs)

# === 基于 marker 表达的自动标签建议（写入 SUBSET_MARKER_COL） ===
# 每个簇找在 marker 库中表达最高的细胞类型——纯计算建议，不自动落地为 final
if _markers and LEIDEN_COL in adata_sub.obs.columns:
    print(f"\n===== marker-based 标签建议（写入 {SUBSET_MARKER_COL}）=====")
    _marker_suggested = {}
    for cl in sorted(adata_sub.obs[LEIDEN_COL].unique(), key=_safe_sort_key):
        _mask = adata_sub.obs[LEIDEN_COL] == cl
        _best_score = -1.0
        _best_type = "Unknown"
        for ct, genes in _markers.items():
            _present_g = [g for g in genes if g in adata_sub.var_names]
            if not _present_g:
                continue
            _expr = adata_sub[_mask][:, _present_g].X
            if sp.issparse(_expr):
                _expr = _expr.toarray()
            _mean_expr = float(_expr.mean())
            if _mean_expr > _best_score:
                _best_score = _mean_expr
                _best_type = ct
        _marker_suggested[str(cl)] = _best_type
        print(f"  Cluster {cl}: → {_best_type} (mean expr={_best_score:.3f})")

    adata_sub.obs[SUBSET_MARKER_COL] = (
        adata_sub.obs[LEIDEN_COL].astype(str).map(_marker_suggested)
    ).astype("category")
    print(f"\nmarker-based 建议已写入: {SUBSET_MARKER_COL}")
else:
    adata_sub.obs[SUBSET_MARKER_COL] = pd.Categorical([np.nan] * adata_sub.n_obs)

# === 初始化 annotation_provenance_subset（与 06 的 provenance 对齐） ===
# provenance 逐簇记录 marker/LLM 建议来源、置信度、PI 决定来源
if SUBSET_PROV_KEY not in adata_sub.uns:
    adata_sub.uns[SUBSET_PROV_KEY] = {}
_prov = adata_sub.uns[SUBSET_PROV_KEY]
for _cl in sorted(adata_sub.obs[LEIDEN_COL].unique(), key=_safe_sort_key):
    _cl_str = str(_cl)
    if _cl_str not in _prov:
        _prov[_cl_str] = {}
    # marker-based 建议
    _prov[_cl_str]["marker_suggested"] = _marker_suggested.get(_cl_str, "") if (_markers and "_marker_suggested" in dir()) else ""
    # LLM 建议（仅 machine-suggested，不是 final）
    _llm_label = ""
    if SUBSET_LLM_COL in adata_sub.obs.columns:
        _cl_llm = adata_sub.obs.loc[adata_sub.obs[LEIDEN_COL] == _cl, SUBSET_LLM_COL].dropna()
        if len(_cl_llm) > 0:
            _llm_label = _cl_llm.mode().iloc[0] if hasattr(_cl_llm, 'mode') else str(_cl_llm.iloc[0])
    _prov[_cl_str]["llm_suggested"] = _llm_label if not (_llm_label or "").startswith("Cluster_") else ""
    _prov[_cl_str]["llm_confidence"] = ""
    _prov[_cl_str].setdefault("pi_decision", "")
    _prov[_cl_str].setdefault("decision_source", "unresolved")  # 默认未决，等 PI 决策

print(f"annotation_provenance_subset 已初始化: {SUBSET_PROV_KEY} ({len(_prov)} 簇)")

### PI 拍板子集精细标签

PI 查看上方 dotplot + LLM 共识结果后，为子集的每个簇填入精细细胞类型标签。
这些标签将回流到主 adata 的 `SUBSET_LABEL_COL`。

**命名建议**：精细标签应比全局标签更具体——
如全局标签 `T_cell` → 精细标签 `CD4_Tcm`、`CD8_Tem`、`Treg` 等。

In [ ]:
# === PI 拍板子集精细标签（镜像 06 cell 30：6 列 schema + provenance 联动） ===
# 支持两种输入方式：
#   方式 1（推荐非 CS 用户）：CSV 文件，6 列
#     cluster, suggested_label, pi_final_label, decision(accept|modify|unresolved), note, annotation_version
#   方式 2（程序员）：直接编辑下方 pi_subset_decisions Python dict
#
# 每簇 decision 必须为 accept / modify / unresolved 之一。
# unresolved 的簇 blockers：annotation gate 会拒绝通过。
#   - accept：LLM 标签可直接采纳
#   - modify：PI 改写了标签，pi_final_label 必须非空
#   - unresolved：未决定，pi_final_label 可以为空，但 subset 闸门不会通过
# annotation_version 必须匹配 SUBSET_OUTPUT_VERSION（防止版本错位）
print("=== PI 拍板：子集精细标签 ===")

_decision_counts = {"accept": 0, "modify": 0, "unresolved": 0}
_allowable_decisions = frozenset(["accept", "modify", "unresolved"])
_version_mismatch_clusters = []
_unresolved_clusters = []
_modify_without_label = []

# ---- 从 SUBSET_PI_CONFIRMATION_CSV 或 pi_subset_decisions 加载 PI 决定 ----
if SUBSET_PI_CONFIRMATION_CSV and os.path.exists(SUBSET_PI_CONFIRMATION_CSV):
    # 方式 1：从 CSV 文件读取（6 列 schema）
    _df_pi = pd.read_csv(SUBSET_PI_CONFIRMATION_CSV, comment="#")
    _required = ["cluster", "suggested_label", "pi_final_label", "decision", "note", "annotation_version"]
    _missing = [col for col in _required if col not in _df_pi.columns]
    if _missing:
        raise ValueError(
            f"subset PI 确认 CSV 缺少必需列: {', '.join(_missing)}；"
            f"需要的 6 列: {', '.join(_required)}。"
            f"当前文件列: {', '.join(_df_pi.columns.tolist())}"
        )
    pi_subset_decisions = {}
    for _, row in _df_pi.iterrows():
        _cl = str(row["cluster"])
        _decision = str(row["decision"]).strip().lower()
        _ver = str(row["annotation_version"]).strip()
        _pi_label = str(row["pi_final_label"]).strip() if pd.notna(row["pi_final_label"]) else ""
        _note = str(row["note"]).strip() if pd.notna(row["note"]) else ""
        pi_subset_decisions[_cl] = {
            "suggested_label": str(row["suggested_label"]).strip() if pd.notna(row["suggested_label"]) else "",
            "pi_final_label": _pi_label,
            "decision": _decision,
            "note": _note,
            "annotation_version": _ver,
        }
    print(f"从 CSV 加载 PI 决定: {SUBSET_PI_CONFIRMATION_CSV} ({len(pi_subset_decisions)} 簇)")
else:
    # 方式 2：Python dict（程序员直接编辑）
    pi_subset_decisions = {
        # "0": {
        #     "suggested_label": "CD4_Tcm",
        #     "pi_final_label": "CD4_Tcm",
        #     "decision": "accept",
        #     "note": "CD4 + CD44 hi + CD62L lo → 效应记忆 T",
        #     "annotation_version": SUBSET_OUTPUT_VERSION,
        # },
        # "1": {
        #     "suggested_label": "CD8_Tem",
        #     "pi_final_label": "CD8_Temra",  # PI 按 CD45RA 表达改写
        #     "decision": "modify",
        #     "note": "GZMK lo + CD45RA hi → Temra 而非 Tem",
        #     "annotation_version": SUBSET_OUTPUT_VERSION,
        # },
        # "2": {
        #     "suggested_label": "Unknown",
        #     "pi_final_label": "",
        #     "decision": "unresolved",
        #     "note": "该簇标记物表达模糊，需进一步 DEG 分析",
        #     "annotation_version": SUBSET_OUTPUT_VERSION,
        # },
        # ... PI 逐簇填入
    }
    print(f"使用 Python dict PI 决定: {len(pi_subset_decisions)} 簇")

# ---- 遍历每簇，验证 decision + 写入 SUBSET_PI_CONFIRMED_COL ----
if pi_subset_decisions and LEIDEN_COL in adata_sub.obs.columns:
    _all_clusters = [str(c) for c in sorted(adata_sub.obs[LEIDEN_COL].unique(), key=_safe_sort_key)]
    _pi_map = {}
    _prov = adata_sub.uns.get(SUBSET_PROV_KEY, {})

    for _cl in _all_clusters:
        _cl_info = pi_subset_decisions.get(_cl, {})
        if not isinstance(_cl_info, dict):
            print(f"  ⚠ Cluster {_cl}: pi_subset_decisions 条目非 dict，跳过")
            continue

        _decision = str(_cl_info.get("decision", "unresolved")).strip().lower()
        _pi_label = str(_cl_info.get("pi_final_label", "")).strip()
        _ver = str(_cl_info.get("annotation_version", "")).strip()
        _suggested = str(_cl_info.get("suggested_label", "")).strip()
        _note = str(_cl_info.get("note", "")).strip()

        # 验证 decision 值
        if _decision not in _allowable_decisions:
            print(f"  ⚠ Cluster {_cl}: 非法的 decision '{_decision}'，回退为 'unresolved'")
            _decision = "unresolved"

        # 版本不匹配 → unresolved
        if _ver != SUBSET_OUTPUT_VERSION:
            _version_mismatch_clusters.append(_cl)
            _decision = "unresolved"
            print(f"  ⚠ Cluster {_cl}: annotation_version '{_ver}' != SUBSET_OUTPUT_VERSION '{SUBSET_OUTPUT_VERSION}' → 强制 unresolved")

        # modify 但无 pi_final_label → unresolved
        if _decision == "modify" and not _pi_label:
            _modify_without_label.append(_cl)
            _decision = "unresolved"
            print(f"  ⚠ Cluster {_cl}: decision=modify 但 pi_final_label 为空 → 强制 unresolved")

        # 计数
        if _decision == "unresolved":
            _unresolved_clusters.append(_cl)
        _decision_counts[_decision] = _decision_counts.get(_decision, 0) + 1

        # 写入 SUBSET_PI_CONFIRMED_COL
        _final_label = _pi_label if _pi_label else _suggested
        _pi_map[_cl] = _final_label if _final_label else np.nan

        # 更新 provenance
        if _cl not in _prov:
            _prov[_cl] = {}
        _prov[_cl]["pi_decision"] = _decision
        _prov[_cl]["pi_final_label"] = _pi_label
        _prov[_cl]["decision_source"] = _decision  # accept | modify | unresolved
        _prov[_cl]["note"] = _note
        _prov[_cl]["annotation_version"] = _ver

    adata_sub.obs[SUBSET_PI_CONFIRMED_COL] = (
        adata_sub.obs[LEIDEN_COL].astype(str).map(_pi_map)
    ).astype("category")
    adata_sub.uns[SUBSET_PROV_KEY] = _prov
    print(f"PI 确认列已写入: {SUBSET_PI_CONFIRMED_COL}")
    print(f"  decisions: accept={_decision_counts['accept']}, modify={_decision_counts['modify']}, unresolved={_decision_counts['unresolved']}")
    if _unresolved_clusters:
        print(f"  unresolved clusters: {_unresolved_clusters}")
        print(f"  → subset 注释闸门将阻止 final 列创建，直到所有簇 resolved")
    if _version_mismatch_clusters:
        print(f"  version mismatch clusters: {_version_mismatch_clusters}")
    if _modify_without_label:
        print(f"  modify 无标签 clusters: {_modify_without_label}")
else:
    adata_sub.obs[SUBSET_PI_CONFIRMED_COL] = pd.Categorical([np.nan] * adata_sub.n_obs)
    print(f"PI 暂未填写子集精细标签，已预建 {SUBSET_PI_CONFIRMED_COL} 空列")

In [ ]:
# === subset 注释闸门（镜像 06 cell 31：逐簇 decision_source 检查） ===
# 闸门逻辑：
#   1. SUBSET_PI_CONFIRMED 必须为 True（PI 主动开关）
#   2. 每簇 decision_source 必须为 accept 或 modify（无一 unresolved）
#   3. SUBSET_ACCEPT_ALL_SUGGESTED 快捷路径：PI 确认"全部接受建议"，
#      则自动为所有未填决定的簇生成 accept 审计记录，同时写回 provenance
# 以上全部满足 → subset_final_gate_passed = True → 创建 cell_type_final_subset_{Vs}
print("\n=== subset 注释闸门 ===")

# ---- 重置 gate ----
subset_final_gate_passed = False

# ---- 前置条件检查 ----
if not SUBSET_PI_CONFIRMED:
    print("subset 闸门 BLOCKED: SUBSET_PI_CONFIRMED=False，PI 尚未确认 subset 注释")
    print("  -> cell_type_final_subset 列不会创建")
    print("  -> 下游可使用 marker/llm_suggested/pi_confirmed 列作为参考")

elif SUBSET_PI_CONFIRMED and SUBSET_ACCEPT_ALL_SUGGESTED:
    # 快捷路径：PI 确认"全部接受建议"——
    # 自动为所有簇生成 accept 审计记录，写回 provenance
    print("subset PI 确认快捷路径: SUBSET_ACCEPT_ALL_SUGGESTED=True")
    print("  为所有簇自动生成 accept 审计记录...")

    _prov = adata_sub.uns.get(SUBSET_PROV_KEY, {})
    _all_clusters = [str(c) for c in sorted(adata_sub.obs[LEIDEN_COL].unique(), key=_safe_sort_key)]
    _pi_map = {}
    _auto_accept_count = 0

    for _cl in _all_clusters:
        _cl_info = pi_subset_decisions.get(_cl, None)
        if isinstance(_cl_info, dict) and _cl_info.get("decision", "").strip().lower() in ("accept", "modify"):
            # 已有明确决定，使用现有决定
            _pi_label = str(_cl_info.get("pi_final_label", "")).strip() or str(_cl_info.get("suggested_label", "")).strip()
            _pi_map[_cl] = _pi_label if _pi_label else np.nan
            continue

        # 无决定 → 自动 accept LLM suggested
        _suggested = ""
        if SUBSET_LLM_COL in adata_sub.obs.columns:
            _cl_llm = adata_sub.obs.loc[adata_sub.obs[LEIDEN_COL] == _cl, SUBSET_LLM_COL].dropna()
            if len(_cl_llm) > 0:
                _suggested = str(_cl_llm.mode().iloc[0]) if hasattr(_cl_llm, 'mode') else str(_cl_llm.iloc[0])

        if _suggested and not _suggested.startswith("Cluster_"):
            _pi_map[_cl] = _suggested
        else:
            # LLM 也无建议 → 回退到 marker suggested
            _pi_map[_cl] = str(_prov.get(_cl, {}).get("marker_suggested", "Unknown"))

        if _cl not in _prov:
            _prov[_cl] = {}
        _prov[_cl]["decision_source"] = "accept"
        _prov[_cl]["pi_decision"] = "accept"
        _prov[_cl]["pi_final_label"] = _pi_map[_cl]
        _prov[_cl]["note"] = f"auto-accepted via SUBSET_ACCEPT_ALL_SUGGESTED ({datetime.datetime.now().isoformat()})"
        _auto_accept_count += 1

    adata_sub.uns[SUBSET_PROV_KEY] = _prov
    # 更新 SUBSET_PI_CONFIRMED_COL（覆盖之前可能不完整的值）
    if _pi_map:
        adata_sub.obs[SUBSET_PI_CONFIRMED_COL] = (
            adata_sub.obs[LEIDEN_COL].astype(str).map(_pi_map)
        ).astype("category")
    print(f"  已自动 accept {_auto_accept_count} 个簇")
    subset_final_gate_passed = True

else:
    # 常规路径：逐簇检查 decision_source
    _prov = adata_sub.uns.get(SUBSET_PROV_KEY, {})
    _all_clusters = [str(c) for c in sorted(adata_sub.obs[LEIDEN_COL].unique(), key=_safe_sort_key)]
    _unresolved = [
        _cl for _cl in _all_clusters
        if _prov.get(_cl, {}).get("decision_source", "unresolved") == "unresolved"
    ]
    if _unresolved:
        print(f"subset 闸门 BLOCKED: {len(_unresolved)} 个簇 unresolved: {_unresolved}")
        print("  -> cell_type_final_subset 列不会创建")
        print("  -> 请在 PI 拍板 cell 中为以上簇填入 decision 或设置 SUBSET_ACCEPT_ALL_SUGGESTED=True")
    else:
        subset_final_gate_passed = True

# ---- 创建 final 列（仅当 gate passed） ----
if subset_final_gate_passed:
    _final_col = f"cell_type_final_subset_{SUBSET_OUTPUT_VERSION}"
    _prov = adata_sub.uns.get(SUBSET_PROV_KEY, {})

    # 从 pi_confirmed 生成 final label
    _map = {}
    for _cl in sorted(adata_sub.obs[LEIDEN_COL].unique(), key=_safe_sort_key):
        _cl_str = str(_cl)
        _prov_entry = _prov.get(_cl_str, {})
        _pi_label = _prov_entry.get("pi_final_label", "")
        _decision = _prov_entry.get("decision_source", "unresolved")
        if _decision in ("accept", "modify") and _pi_label:
            _map[_cl_str] = _pi_label
        else:
            # accept 且 pi_final_label 为空：回退至 LLM suggested → marker suggested
            if SUBSET_LLM_COL in adata_sub.obs.columns:
                _cl_llm = adata_sub.obs.loc[adata_sub.obs[LEIDEN_COL] == _cl, SUBSET_LLM_COL].dropna()
                if len(_cl_llm) > 0:
                    _llm_val = str(_cl_llm.mode().iloc[0]) if hasattr(_cl_llm, 'mode') else str(_cl_llm.iloc[0])
                    if _llm_val and not _llm_val.startswith("Cluster_"):
                        _map[_cl_str] = _llm_val
                        continue
            _map[_cl_str] = _prov_entry.get("marker_suggested", "Unknown")

    adata_sub.obs[_final_col] = (
        adata_sub.obs[LEIDEN_COL].astype(str).map(_map)
    ).astype("category")

    # 记录 annotation_gate 元数据（与 06 对齐）
    adata_sub.uns["annotation_gate"] = {
        "gate_passed": True,
        "timestamp": datetime.datetime.now().isoformat(),
        "total_clusters": len(_all_clusters),
        "decisions": {
            "accept": sum(1 for _cl in _all_clusters if _prov.get(_cl, {}).get("decision_source") == "accept"),
            "modify": sum(1 for _cl in _all_clusters if _prov.get(_cl, {}).get("decision_source") == "modify"),
            "unresolved": 0,
        },
        "accept_all_suggested": SUBSET_ACCEPT_ALL_SUGGESTED,
    }

    print(f"\n✓ subset 注释闸门通过！cell_type_final_subset_{SUBSET_OUTPUT_VERSION} 已创建")
    print(f"  列: {_final_col}")
    print(f"  标签种类: {adata_sub.obs[_final_col].nunique()}")
    print(f"  标签分布: {dict(adata_sub.obs[_final_col].value_counts())}")

print(f"\nsubset_final_gate_passed = {subset_final_gate_passed}")

## 标签回流：将子集精细标签写回主 adata

这是 06c 最关键的一步。子集的精细标签只有回流到主 adata 的
`SUBSET_LABEL_COL` 写回后，下游才能同时利用两个层级：

- `UPSTREAM_LABEL_COL`：大类标签（所有细胞都有）
- `SUBSET_LABEL_COL`：精细标签（仅子集细胞有值，其余 NaN）

**为什么不动 `UPSTREAM_LABEL_COL`？**
这是 SPEC 835 的硬约束——两个粒度的标签共存而不互相污染。
大类分析用 `UPSTREAM_LABEL_COL`；精细分析用 `SUBSET_LABEL_COL` 过滤 NaN。

**为什么主 adata 回写版本号递增（v1→v2）？**
回流修改了主 adata 的 obs 列——这是一个新的数据状态，
应按版本化约定创建新版本而非覆盖原版。下游 notebook 指定
合适的版本号即可。

In [ ]:
# === 回流闸门（决策6 的 6e：subset 注释确认后，主对象回流须 PI 额外确认 + 版本 bump） ===
# 回流闸门逻辑（三个条件必须全部满足）：
#   1. subset_final_gate_passed = True  —— subset 注释闸门已通过
#   2. MAIN_REFLOW_CONFIRMED = True     —— PI 主动开关：确认 subset 标签可以写回主对象
#   3. MAIN_OUTPUT_VERSION != UPSTREAM_LABEL_VERSION —— 主版本号必须刻意 bump（解耦），
#      防止默认 v1==v1 时意外覆盖上游
# 不通过原因透明输出，方便 PI 排查
print("\n=== 回流闸门 ===")

main_reflow_gate_passed = False  # 重置

if not subset_final_gate_passed:
    print("回流闸门 BLOCKED: subset 注释闸门未通过 (subset_final_gate_passed=False)")
    print("  -> 主对象不会加载或修改")

elif not MAIN_REFLOW_CONFIRMED:
    print("回流闸门 BLOCKED: MAIN_REFLOW_CONFIRMED=False，PI 尚未确认回流操作")
    print("  -> 仅子集 checkpoint 产出，主对象保持不变")

elif MAIN_OUTPUT_VERSION == UPSTREAM_LABEL_VERSION:
    print(f"回流闸门 BLOCKED: MAIN_OUTPUT_VERSION ({MAIN_OUTPUT_VERSION}) == UPSTREAM_LABEL_VERSION ({UPSTREAM_LABEL_VERSION})")
    print("  -> 主版本号必须刻意 bump，防止默认等号时意外覆盖上游")
    print("  -> 请在 PARAMS cell 中将 MAIN_OUTPUT_VERSION 改为与 UPSTREAM_LABEL_VERSION 不同的值")

else:
    main_reflow_gate_passed = True

print(f"main_reflow_gate_passed = {main_reflow_gate_passed}")
if main_reflow_gate_passed:
    print(f"  MAIN_OUTPUT_VERSION ({MAIN_OUTPUT_VERSION}) != UPSTREAM_LABEL_VERSION ({UPSTREAM_LABEL_VERSION}) ✓")
    print(f"  将加载主对象并执行回流...")

In [ ]:
# 标签回流：将子集精细标签写回主 adata（仅当回流闸门通过）。
# 回流闸门不通过时，整段跳过：主 adata 不加载、不修改。
print("=== 标签回流 ===")

if main_reflow_gate_passed:
    # 重新加载主 adata（上游 path——而非更新后的版本，避免循环）
    print(f"加载主 adata: {_upstream_checkpoint}")
    main_adata = sc.read_h5ad(_upstream_checkpoint)

    # 回流前快照：保存 UPSTREAM_LABEL_COL 用于回流后真实验证
    _check = main_adata.obs[UPSTREAM_LABEL_COL].copy()

    # 初始化 SUBSET_LABEL_COL——全部 NaN
    main_adata.obs[SUBSET_LABEL_COL] = np.nan

    # 对齐：子集细胞 → 主 adata 中的对应细胞
    _sub_labels = adata_sub.obs[SUBSET_LABEL_COL]
    _common_idx = main_adata.obs_names.intersection(adata_sub.obs_names)
    print(f"可回流细胞: {len(_common_idx):,} / {adata_sub.n_obs:,} 子集细胞")

    # 将精细标签写入主 adata 对应细胞
    main_adata.obs.loc[_common_idx, SUBSET_LABEL_COL] = (
        _sub_labels.loc[_common_idx]
    )

    # 处理缺失（子集中有但主 adata 中没有的细胞——不应发生，但防御处理）
    _missing = set(adata_sub.obs_names) - set(_common_idx)
    if _missing:
        print(f"⚠ 警告: {len(_missing)} 个子集细胞不在主 adata 中，无法回流")

    _n_reflowed = main_adata.obs[SUBSET_LABEL_COL].notna().sum()
    print(f"已回流: {_n_reflowed:,} 细胞 -> obs[{SUBSET_LABEL_COL!r}]")
    print(f"  精细标签种类: {main_adata.obs[SUBSET_LABEL_COL].nunique()}")

    # 确认 UPSTREAM_LABEL_COL 未被修改——before/after 真实验证
    assert (_check == main_adata.obs[UPSTREAM_LABEL_COL]).all(), (
        f"{UPSTREAM_LABEL_COL} 在回流过程中被意外修改！"
    )
    print(f"  {UPSTREAM_LABEL_COL} 保持不变（before/after 真实验证通过）")

    # 记录回流元数据
    main_adata.uns[f"{SUBSET_LABEL_COL}_notes"] = {
        "source": "06c_subset",
        "subset_labels": list(SUBSET_LABELS),
        "subset_file": f"06c_subset_{SUBSET_OUTPUT_VERSION}.h5ad",
        "n_cells_refined": int(_n_reflowed),
        "refined_cell_types": sorted(
            main_adata.obs[SUBSET_LABEL_COL].dropna().unique().tolist()
        ),
        "note": (
            f"{UPSTREAM_LABEL_COL} retains broad labels; {SUBSET_LABEL_COL} provides fine labels. "
            "Downstream analyses choose the appropriate column."
        ),
        "timestamp": datetime.datetime.now().isoformat(),
    }
else:
    print("回流闸门未通过，跳过主对象加载与回流。")
    main_adata = None

In [ ]:
# === 构建统一精细标签列（仅当回流成功且 main_adata 非空） ===
# 将 UPSTREAM_LABEL_COL（粗）和 SUBSET_LABEL_COL（精细）合并为 MAIN_LABEL_COL
# 规则：有 subset 精细标签的细胞用精细标签，其余用粗标签
if main_reflow_gate_passed and main_adata is not None:
    _unified_col = MAIN_LABEL_COL
    main_adata.obs[_unified_col] = main_adata.obs[UPSTREAM_LABEL_COL].copy()

    # 用 subset 精细标签覆盖（仅对子集细胞）
    _has_subset = main_adata.obs[SUBSET_LABEL_COL].notna()
    main_adata.obs.loc[_has_subset, _unified_col] = main_adata.obs.loc[_has_subset, SUBSET_LABEL_COL]

    _n_refined = _has_subset.sum()
    print(f"✓ 统一标签列 '{_unified_col}' 已创建")
    print(f"  {_n_refined:,} 细胞使用 subset 精细标签")
    print(f"  {(~_has_subset).sum():,} 细胞保持粗标签")
    print(f"  → 下游分析（07_downstream）可直接使用此列作为 groupby")
else:
    print("回流闸门未通过，跳过统一标签列创建。")

## 写出产物

两个 h5ad 文件：
1. **子集 checkpoint**：写入本次 run 的 draft
2. **主对象回流 artifact**：与 checkpoint 同属一个 draft

两个文件命名均遵循版本化约定。

In [ ]:
# === 双 H5AD draft 输出合同（含闸门分支） ===
# 闸门分支：
#   - subset_final_gate_passed=True  → 双产物：subset checkpoint + main reflow
#   - subset_final_gate_passed=False → 单产物：subset draft only（NEEDS_REVIEW，无 main reflow）
# 回流闸门未通过时 main_adata=None，_output_contract 不可调用（依赖 main.obs）
_OUTPUT_VERSION_RE = re.compile(r"v[1-9][0-9]*\\Z")
_H5AD_BASENAME_RE = re.compile(r"[A-Za-z0-9][A-Za-z0-9._-]*\\.h5ad\\Z")

def _series_equal(left, right):
    if not left.index.equals(right.index) or not left.isna().equals(right.isna()): return False
    present = ~left.isna()
    return left.loc[present].astype("string").equals(right.loc[present].astype("string"))

def _output_contract(subset, main):
    subset_name = f"06c_subset_{SUBSET_OUTPUT_VERSION}.h5ad"
    main_name = f"06_annotated_{MAIN_OUTPUT_VERSION}.h5ad"
    names = (subset_name, main_name)
    versions_safe = all(isinstance(v, str) and _OUTPUT_VERSION_RE.fullmatch(v) for v in (SUBSET_OUTPUT_VERSION, MAIN_OUTPUT_VERSION))
    names_safe = all(Path(name).name == name and name.isascii() and _H5AD_BASENAME_RE.fullmatch(name) for name in names)
    common = main.obs_names.intersection(subset.obs_names)
    aligned = subset.obs_names.equals(common) and subset.obs_names.is_unique and main.obs_names.is_unique
    source_aligned = main.obs_names.equals(_verified_upstream_index)
    coarse_unchanged = source_aligned and UPSTREAM_LABEL_COL in main.obs and _series_equal(main.obs[UPSTREAM_LABEL_COL], _verified_upstream_coarse)
    expected_columns = set(_verified_upstream_obs_columns) | {SUBSET_LABEL_COL, MAIN_LABEL_COL}
    columns_exact = set(main.obs.columns) == expected_columns
    subset_labels_complete = SUBSET_LABEL_COL in subset.obs and subset.obs[SUBSET_LABEL_COL].notna().all()
    fine_aligned = aligned and SUBSET_LABEL_COL in main.obs and subset_labels_complete and _series_equal(main.obs.loc[subset.obs_names, SUBSET_LABEL_COL], subset.obs[SUBSET_LABEL_COL])
    outside = main.obs_names.difference(subset.obs_names)
    fine_only_subset = SUBSET_LABEL_COL in main.obs and main.obs.loc[outside, SUBSET_LABEL_COL].isna().all()
    unified_inside = MAIN_LABEL_COL in main.obs and subset_labels_complete and _series_equal(main.obs.loc[subset.obs_names, MAIN_LABEL_COL], subset.obs[SUBSET_LABEL_COL])
    unified_outside = MAIN_LABEL_COL in main.obs and _series_equal(main.obs.loc[outside, MAIN_LABEL_COL], main.obs.loc[outside, UPSTREAM_LABEL_COL])
    gates = {"versions_safe": versions_safe, "main_version_new": MAIN_OUTPUT_VERSION != UPSTREAM_LABEL_VERSION,
             "filenames_safe_distinct": names_safe and len(set(names)) == 2, "subset_nonempty": 0 < subset.n_obs < main.n_obs,
             "index_aligned": aligned and source_aligned, "coarse_labels_unchanged": coarse_unchanged, "main_obs_columns_exact": columns_exact,
             "fine_labels_aligned": fine_aligned, "fine_labels_only_subset": fine_only_subset,
             "unified_matches_fine_inside": unified_inside, "unified_matches_coarse_outside": unified_outside}
    gates = {key: bool(value) for key, value in gates.items()}
    return names, gates

def _manifest_base(gates):
    return {"run_id": RUN_ID, "stage": "06c_subset", "inputs": [
        {"path": str(_upstream_manifest_path), "sha256": _upstream_manifest_sha256},
        {"path": str(_upstream_checkpoint), "sha256": _upstream_checkpoint_sha256}],
        "effective_parameters": snapshot_effective_parameters(globals(), path_root=_root),
        "runtime_provenance": collect_runtime_provenance(_root, ("scanpy", "anndata")),
        "hard_postconditions": gates, "warnings": []}

def _record_output_failure(paths, outputs, error):
    for output in outputs: Path(output).unlink(missing_ok=True)
    base = {"run_id": RUN_ID, "stage": "06c_subset", "stage_status": "FAILED",
            "failure": {"type": type(error).__name__, "message": str(error)}}
    atomic_write_json(paths.manifest_path, base, overwrite=paths.manifest_path.exists())

def _complete_output_write(subset, main, names, gates, paths, outputs, base):
    if not all(gates.values()): raise ValueError(f"Stage 06c output gates FAILED: {gates}")
    subset_path, main_path = outputs
    upstream = {"run_id": UPSTREAM_RUN_ID, "manifest": str(_upstream_manifest_path), "manifest_sha256": _upstream_manifest_sha256,
                "checkpoint": str(_upstream_checkpoint), "checkpoint_sha256": _upstream_checkpoint_sha256}
    subset.uns.update({"stage": "06c_subset", "status": "NEEDS_REVIEW", "run_id": RUN_ID,
                       "versions": {"upstream_label": UPSTREAM_LABEL_VERSION, "subset_output": SUBSET_OUTPUT_VERSION, "main_output": MAIN_OUTPUT_VERSION}, "upstream": upstream})
    main.uns[f"06c_subset_reflow_{MAIN_OUTPUT_VERSION}"] = {"run_id": RUN_ID, "status": "NEEDS_REVIEW", "subset_file": names[0],
        "subset_label_col": SUBSET_LABEL_COL, "main_label_col": MAIN_LABEL_COL, "subset_labels": list(SUBSET_LABELS),
        "versions": {"subset": SUBSET_OUTPUT_VERSION, "main": MAIN_OUTPUT_VERSION}, "upstream": upstream}
    if any(path.exists() for path in outputs): raise FileExistsError("draft output collision")
    subset.write_h5ad(subset_path, compression="lzf"); main.write_h5ad(main_path, compression="lzf")
    subset_hash, main_hash = sha256_file(subset_path), sha256_file(main_path)
    manifest = dict(base); manifest.update({"stage_status": "NEEDS_REVIEW", "checkpoint": {"path": names[0], "sha256": subset_hash},
        "artifacts": [{"role": "subset", "path": names[0], "sha256": subset_hash}, {"role": "main_reflow", "path": names[1], "sha256": main_hash}]})
    atomic_write_json(paths.manifest_path, manifest); validate_checkpoint(paths.manifest_path); validate_artifacts(paths.manifest_path)

def _write_06c_outputs(subset, main):
    names, gates = _output_contract(subset, main)
    base = _manifest_base(gates)
    paths = prepare_run(RUN_ROOT, RUN_ID)
    try: outputs = [paths.draft_dir / name for name in names]; _complete_output_write(subset, main, names, gates, paths, outputs, base)
    except Exception as error: _record_output_failure(paths, outputs, error); raise
    return paths, outputs[0], outputs[1]

def _write_06c_subset_draft(subset):
    """仅输出子集 checkpoint（单产物），gate 未通过时使用。

    subset 注释闸门未通过时，不创建 cell_type_final_subset 列，
    仅将 marker + LLM + pi_confirmed 三层写入 subset h5ad，
    status=NEEDS_REVIEW，留给 PI 审查后决定。
    无 main reflow artifact（主对象不受影响）。
    """
    subset_name = f"06c_subset_{SUBSET_OUTPUT_VERSION}.h5ad"
    if not (_OUTPUT_VERSION_RE.fullmatch(SUBSET_OUTPUT_VERSION) and
            Path(subset_name).name == subset_name and subset_name.isascii() and
            _H5AD_BASENAME_RE.fullmatch(subset_name)):
        raise ValueError("subset 文件名/版本不安全")

    base = {"run_id": RUN_ID, "stage": "06c_subset", "inputs": [
        {"path": str(_upstream_manifest_path), "sha256": _upstream_manifest_sha256},
        {"path": str(_upstream_checkpoint), "sha256": _upstream_checkpoint_sha256}],
        "effective_parameters": snapshot_effective_parameters(globals(), path_root=_root),
        "runtime_provenance": collect_runtime_provenance(_root, ("scanpy", "anndata")),
        "hard_postconditions": {"subset_final_gate_passed": False,
                                "subset_pi_confirmed": bool(SUBSET_PI_CONFIRMED),
                                "main_reflow_gate_passed": False},
        "warnings": [],
        "gate_status": {
            "subset_final_gate_passed": False,
            "subset_pi_confirmed": bool(SUBSET_PI_CONFIRMED),
            "note": "subset 注释闸门未通过，仅产出 draft（无 cell_type_final_subset 列）。PI 审查后可重新设置 SUBSET_PI_CONFIRMED=True 再跑以创建 final。",
        }}

    paths = prepare_run(RUN_ROOT, RUN_ID)
    subset_path = paths.draft_dir / subset_name

    try:
        if subset_path.exists(): raise FileExistsError("draft output collision")
        upstream = {"run_id": UPSTREAM_RUN_ID, "manifest": str(_upstream_manifest_path),
                    "manifest_sha256": _upstream_manifest_sha256,
                    "checkpoint": str(_upstream_checkpoint),
                    "checkpoint_sha256": _upstream_checkpoint_sha256}
        subset.uns.update({"stage": "06c_subset", "status": "NEEDS_REVIEW", "run_id": RUN_ID,
                           "versions": {"upstream_label": UPSTREAM_LABEL_VERSION,
                                        "subset_output": SUBSET_OUTPUT_VERSION,
                                        "main_output": MAIN_OUTPUT_VERSION},
                           "upstream": upstream})
        subset.write_h5ad(subset_path, compression="lzf")
        subset_hash = sha256_file(subset_path)
        manifest = dict(base)
        manifest.update({"stage_status": "NEEDS_REVIEW",
                         "checkpoint": {"path": subset_name, "sha256": subset_hash},
                         "artifacts": [{"role": "subset_draft", "path": subset_name, "sha256": subset_hash}]})
        atomic_write_json(paths.manifest_path, manifest)
        validate_checkpoint(paths.manifest_path)
        validate_artifacts(paths.manifest_path)
    except Exception as error:
        _record_output_failure(paths, [subset_path], error)
        raise

    return paths, subset_path

# === 闸门分支：选择双产物 or 单产物路径 ===
if subset_final_gate_passed and main_reflow_gate_passed and main_adata is not None:
    # 双闸门通过：subset checkpoint + main reflow
    run_paths, draft_subset_path, draft_main_path = _write_06c_outputs(adata_sub, main_adata)
    print(f"Stage 06c: NEEDS_REVIEW；双产物保留在 {run_paths.draft_dir}，本 PR 不提升。")
    print(f"  子集: {draft_subset_path}")
    print(f"  回流: {draft_main_path}")
else:
    # subset 闸门未通过 or 回流闸门未通过：仅子集 draft
    run_paths, draft_subset_path = _write_06c_subset_draft(adata_sub)
    draft_main_path = None  # 无回流产物
    print(f"Stage 06c: NEEDS_REVIEW（闸门未通过，仅子集 draft）")
    print(f"  subset_final_gate_passed={subset_final_gate_passed}")
    print(f"  main_reflow_gate_passed={main_reflow_gate_passed}")
    print(f"  子集 draft: {draft_subset_path}")
    if not subset_final_gate_passed:
        print(f"  → PI 审查后可设置 SUBSET_PI_CONFIRMED=True 再跑以创建 final")
    elif not main_reflow_gate_passed:
        print(f"  → 子集注释已确认，PI 审查后可设置 MAIN_REFLOW_CONFIRMED=True 并 bump MAIN_OUTPUT_VERSION 再跑以回流")

### 下一步：对子集内精细簇做深度剖析

06c 完成了 subset 重聚类和注释。如需对 subset 内的每个精细簇做逐簇深度解读（DEG、邻居对比、LLM 叙述），请运行 `06b_per_cluster.ipynb` 的 **subset 模式**。


In [ ]:
# === 下一步引导（闸门感知） ===
print("===== 06c 完成 → 下一步建议 =====\n")
print(f"子集产出: {draft_subset_path}")
print(f"子集注释列: {SUBSET_LABEL_COL}")
print(f"子集包含 {adata_sub.n_obs:,} cells, {adata_sub.obs[LEIDEN_COL].nunique()} clusters\n")

if not subset_final_gate_passed:
    print("⚠️ subset 注释闸门未通过，cell_type_final_subset 列未创建。")
    print("   PI 审查后请执行以下操作之一：")
    print("   A) 在 PI 拍板 cell 中为每个簇填入 decision")
    print("   B) 设置 SUBSET_ACCEPT_ALL_SUGGESTED=True 批量接受")
    print("   C) 设置 SUBSET_PI_CONFIRMED=True 并重新 Run All\n")
elif not main_reflow_gate_passed:
    print("⚠️ subset 注释已确认，但回流闸门未通过。")
    print("   请将 MAIN_REFLOW_CONFIRMED 改为 True 并确保 MAIN_OUTPUT_VERSION != UPSTREAM_LABEL_VERSION")
    print("   然后重新 Run All 以回流标签至主对象。\n")
else:
    if draft_main_path:
        print(f"主对象回流产出: {draft_main_path}")
    print(f"统一标签列: {MAIN_LABEL_COL}\n")

print("如需对这些精细簇做逐簇深度剖析（DEG + 邻居对比 + LLM 叙述），")
print("请打开 06b_per_cluster.ipynb，改 PARAMS 为 subset 模式:\n")
print(f'    UPSTREAM_RUN_ID = "<06c promoted run id>"')
print(f'    LABEL_COL = "{SUBSET_LABEL_COL}"')
print(f'    MODE = "subset"')
print(f'    RUN_ROOT = "results/runs"')
print(f'    REPORT_DIRNAME = "reports"\n')
print("然后 Run All。06b 会对 subset 内的每个精细簇产出独立报告。")

### Stage 06c Verdict

本 stage 完成后应确认：
- [ ] subset 精细注释完成（`SUBSET_LABEL_COL` 已填入）
- [ ] 统一标签列 `MAIN_LABEL_COL` 已生成
- [ ] 如需对 subset 内簇做深度剖析 -> 转 06b(subset 模式)


In [ ]:
# 内存纪律——del + gc 释放跨越 stage 边界。
# 为什么必须释放？子集 adata 虽比主 adata 小，但连同 embedding
# + obsm 矩阵仍可达到数 GB。不及时释放会累积到下游 OOM。
del adata_sub
if main_adata is not None:
    del main_adata
gc.collect()
print("内存已释放")